# 04 - Machine Learning Prediction of the Transport Descriptor

## Aim of this notebook

In this notebook, we use the simulation-based hydrogen-defect dataset generated in Notebook 3 to train machine-learning regression models.

The target variable is the transport-related descriptor:

```text
T_score = 1 / (1 + mean_IPR + defect_density)
```

Because this target is explicitly defined using `mean_ipr` and `defect_density`, we evaluate two machine-learning settings:

1. Full-feature surrogate model
2. Reduced-feature prediction model

This separation allows us to distinguish between learning a designed mathematical proxy and predicting the proxy from more indirect physical descriptors.


## Mathematical Formulas Used in This Notebook

The machine-learning problem is written as a supervised regression task:

$$
\mathbf{x}_i \rightarrow y_i,
$$

where $\mathbf{x}_i$ is the feature vector for sample $i$, and $y_i$ is the target transport descriptor.

The model learns a function

$$
\hat{y}_i = f(\mathbf{x}_i),
$$

where $\hat{y}_i$ is the predicted transport descriptor.

For linear regression-style intuition, this can be written as

$$
\hat{y}_i = w_0 + \sum_{j=1}^{p} w_j x_{ij}.
$$

The mean absolute error is

$$
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|.
$$

The root mean squared error is

$$
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}.
$$

The coefficient of determination is

$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\overline{y})^2}.
$$

A higher $R^2$ means that the model explains more of the variation in the transport descriptor.


## How to read this notebook

This notebook trains machine-learning models to predict the transport descriptor from physics-inspired features.

There are two modeling strategies:

1. **Full-feature surrogate model**: uses all available descriptors.
2. **Reduced-feature prediction model**: uses a smaller, more interpretable feature set.

The key question is:  
Can a machine-learning model learn the relationship between hydrogen-defect descriptors and transport behavior?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

### Mathematical meaning of train-test splitting

The dataset is separated into training and testing subsets:

$$
\mathcal{D}=\mathcal{D}_{\mathrm{train}}\cup\mathcal{D}_{\mathrm{test}},
\qquad
\mathcal{D}_{\mathrm{train}}\cap\mathcal{D}_{\mathrm{test}}=\emptyset.
$$

The model learns from $\mathcal{D}_{\mathrm{train}}$ and is evaluated on unseen samples in $\mathcal{D}_{\mathrm{test}}$.


### Mathematical meaning of the evaluation metrics

The errors are computed from the true target $y_i$ and prediction $\hat{y}_i$:

$$
\mathrm{MAE}=\frac{1}{n}\sum_i |y_i-\hat{y}_i|,
\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2}.
$$

The $R^2$ score is

$$
R^2=1-\frac{\sum_i(y_i-\hat{y}_i)^2}{\sum_i(y_i-\overline{y})^2}.
$$


### Explanation of the code cell above

This cell imports the libraries needed for machine learning.

- `pandas` and `numpy` handle data.
- `matplotlib` creates plots.
- `train_test_split` separates training and test data.
- `RandomForestRegressor` and `GradientBoostingRegressor` are tree-based regression models.
- `mean_absolute_error`, `mean_squared_error`, and `r2_score` evaluate prediction quality.
- `XGBRegressor` is used only if XGBoost is installed.

The `try/except` block keeps the notebook usable even when XGBoost is unavailable.


## Load Dataset

The dataset contains defect-design descriptors, spectral descriptors, localization descriptors, and the transport-related target value.

Each row corresponds to one random hydrogen-defect configuration.

In [ ]:
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
FIGURE_DIR = ROOT / 'figures'
RESULTS_DIR = ROOT / 'results'

FIGURE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / 'hydrogen_defect_dataset.csv'
dataset = pd.read_csv(dataset_path)

dataset.head()

### Explanation of the code cell above

This cell loads the dataset generated in Notebook 3.

The file `hydrogen_defect_dataset.csv` contains simulated defect configurations and their transport descriptors.

The notebook also creates `figures` and `results` folders. Figures will store plots, and results will store model-comparison tables.


## Quick Dataset Check

In [ ]:
dataset.shape

### Explanation of the code cell above

This cell checks the dataset dimensions.

It tells you how many simulated samples and how many variables are available.

This is a basic but important sanity check before training any model.


In [ ]:
dataset.describe()

### Explanation of the code cell above

This cell summarizes the numerical variables.

It helps you inspect the range of each feature and the target. For example, you can check whether values are extremely small, extremely large, or unexpectedly constant.

This step is useful for detecting data problems before model training.


In [ ]:
dataset.isna().sum()

### Explanation of the code cell above

This cell checks missing values in each column.

Missing values can cause machine-learning models to fail or behave unexpectedly.

In this dataset, `clustering_index` may be missing for single-defect configurations because clustering requires at least two defects.


Single-defect configurations do not have a physically defined clustering relationship with other defects. For machine-learning compatibility, missing clustering values are filled with `0.0`. This means that single-defect cases are treated as non-clustered configurations in the regression model.

In [ ]:
dataset_ml = dataset.copy()
dataset_ml['clustering_index'] = dataset_ml['clustering_index'].fillna(0.0)

### Explanation of the code cell above

This cell prepares the dataset for machine learning by filling missing `clustering_index` values with `0.0`.

This choice means:

- if there is only one defect, there is no clustering pattern,
- therefore the clustering contribution is treated as zero.

This is a practical modeling choice. In a publication, you should explicitly mention how missing clustering values were handled.


## Notebook 4A - Full-Feature Surrogate Model

In this experiment, we use the full feature set, including `mean_ipr` and `defect_density`.

Because the transport descriptor is explicitly defined as:

```text
T_score = 1 / (1 + mean_IPR + defect_density)
```

this experiment is best interpreted as a surrogate-modeling baseline.

The goal is not to claim that the model discovered a new physical law. Instead, the goal is to verify that machine learning can reproduce a designed physics-informed transport proxy from physically meaningful descriptors.

In [ ]:
full_features = [
    'defect_density',
    'defect_strength',
    'mean_defect_distance',
    'clustering_index',
    'energy_gap',
    'energy_bandwidth',
    'mean_ipr',
    'max_ipr',
]

target = 'transport_descriptor'

X_full = dataset_ml[full_features]
y = dataset_ml[target]

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full,
    y,
    test_size=0.2,
    random_state=42,
)

### Mathematical meaning of train-test splitting

The dataset is separated into training and testing subsets:

$$
\mathcal{D}=\mathcal{D}_{\mathrm{train}}\cup\mathcal{D}_{\mathrm{test}},
\qquad
\mathcal{D}_{\mathrm{train}}\cap\mathcal{D}_{\mathrm{test}}=\emptyset.
$$

The model learns from $\mathcal{D}_{\mathrm{train}}$ and is evaluated on unseen samples in $\mathcal{D}_{\mathrm{test}}$.


### Explanation of the code cell above

This cell defines the full-feature machine-learning experiment.

The model receives both defect-design descriptors and physics-derived descriptors, such as energy bandwidth and IPR.

The data is split into:

- training set: used to fit the model,
- test set: used to evaluate generalization.

The target is `transport_descriptor`, meaning the model learns to predict transport behavior from the input descriptors.


## Reusable Model Training Functions

In [ ]:
def make_models():
    models = {
        'Random Forest': RandomForestRegressor(
            n_estimators=300,
            random_state=42,
        ),
        'Gradient Boosting': GradientBoostingRegressor(
            random_state=42,
        ),
    }

    if XGBOOST_AVAILABLE:
        models['XGBoost'] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            random_state=42,
        )

    return models


def evaluate_regression_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return {
        'model': model,
        'y_test': y_test,
        'y_pred': y_pred,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
    }

### Mathematical meaning of the evaluation metrics

The errors are computed from the true target $y_i$ and prediction $\hat{y}_i$:

$$
\mathrm{MAE}=\frac{1}{n}\sum_i |y_i-\hat{y}_i|,
\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2}.
$$

The $R^2$ score is

$$
R^2=1-\frac{\sum_i(y_i-\hat{y}_i)^2}{\sum_i(y_i-\overline{y})^2}.
$$


### Explanation of the code cell above

This cell defines reusable model-training functions.

`make_models` creates the regression models.  
`evaluate_regression_model` trains a model and calculates evaluation metrics.

The metrics mean:

- **MAE**: average absolute prediction error,
- **RMSE**: error metric that penalizes larger mistakes more strongly,
- **R²**: fraction of target variance explained by the model.

Reusable functions make the notebook cleaner and reduce repeated code.


## Train Notebook 4A Models

In [ ]:
full_results = {}

for name, model in make_models().items():
    full_results[name] = evaluate_regression_model(
        model,
        X_train_full,
        X_test_full,
        y_train_full,
        y_test_full,
    )

full_results_table = pd.DataFrame(
    [
        {
            'experiment': 'Full-feature surrogate',
            'model': name,
            'MAE': result['MAE'],
            'RMSE': result['RMSE'],
            'R2': result['R2'],
        }
        for name, result in full_results.items()
    ]
)

full_results_table

### Explanation of the code cell above

This cell trains and evaluates the full-feature models.

Each model is fitted on the training data and evaluated on the test data. The results are stored in `full_results_table`.

The purpose is to see which algorithm best learns the mapping from physics-inspired features to the transport descriptor when all features are available.


## Notebook 4B - Reduced-Feature Prediction Model

In this second experiment, we remove the direct target-defining features:

- `mean_ipr`
- `defect_density`

This makes the problem harder and scientifically more interesting.

The goal is to test whether the transport proxy can be predicted from more indirect descriptors such as defect strength, defect spacing, clustering, spectral bandwidth, and maximum localization.

This experiment asks:

> Can the model infer transport-related behavior without directly seeing the variables used in the target formula?

In [ ]:
reduced_features = [
    'defect_strength',
    'mean_defect_distance',
    'clustering_index',
    'energy_gap',
    'energy_bandwidth',
    'max_ipr',
]

X_reduced = dataset_ml[reduced_features]

X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42,
)

### Mathematical meaning of train-test splitting

The dataset is separated into training and testing subsets:

$$
\mathcal{D}=\mathcal{D}_{\mathrm{train}}\cup\mathcal{D}_{\mathrm{test}},
\qquad
\mathcal{D}_{\mathrm{train}}\cap\mathcal{D}_{\mathrm{test}}=\emptyset.
$$

The model learns from $\mathcal{D}_{\mathrm{train}}$ and is evaluated on unseen samples in $\mathcal{D}_{\mathrm{test}}$.


### Explanation of the code cell above

This cell defines a reduced-feature experiment.

The reduced feature set removes some variables and keeps a smaller group of physically meaningful descriptors.

This is useful because a simpler model can be easier to interpret. If the reduced model performs similarly to the full model, it means many features may be unnecessary.


## Train Notebook 4B Models

In [ ]:
reduced_results = {}

for name, model in make_models().items():
    reduced_results[name] = evaluate_regression_model(
        model,
        X_train_reduced,
        X_test_reduced,
        y_train_reduced,
        y_test_reduced,
    )

reduced_results_table = pd.DataFrame(
    [
        {
            'experiment': 'Reduced-feature prediction',
            'model': name,
            'MAE': result['MAE'],
            'RMSE': result['RMSE'],
            'R2': result['R2'],
        }
        for name, result in reduced_results.items()
    ]
)

reduced_results_table

### Explanation of the code cell above

This cell trains and evaluates the reduced-feature models.

The same algorithms and metrics are used as in the full-feature experiment. This makes the comparison fair.

The output table shows whether the reduced feature set keeps enough information to predict the transport descriptor accurately.


## Compare Notebook 4A and Notebook 4B

The full-feature surrogate model is expected to perform better because it includes the variables used directly in the transport descriptor formula.

The reduced-feature model is expected to perform worse, but it provides a more meaningful test of whether indirect physical descriptors carry information about the transport proxy.

In [ ]:
comparison_table = pd.concat(
    [full_results_table, reduced_results_table],
    ignore_index=True,
)

comparison_path = RESULTS_DIR / 'notebook4_model_comparison.csv'
comparison_table.to_csv(comparison_path, index=False)

comparison_table

### Explanation of the code cell above

This cell combines the full-feature and reduced-feature results into one comparison table.

It also saves the table as:

`notebook4_model_comparison.csv`

This saved result is useful for reports, papers, or later notebooks because it records which model performed best.


## Actual vs Predicted Plot

In [ ]:
best_row = comparison_table.sort_values('R2', ascending=False).iloc[0]
best_experiment = best_row['experiment']
best_model_name = best_row['model']

if best_experiment == 'Full-feature surrogate':
    best_result = full_results[best_model_name]
else:
    best_result = reduced_results[best_model_name]

y_test_best = best_result['y_test']
y_pred_best = best_result['y_pred']

plt.figure(figsize=(6, 6))
plt.scatter(y_test_best, y_pred_best, alpha=0.7)
plt.plot(
    [y_test_best.min(), y_test_best.max()],
    [y_test_best.min(), y_test_best.max()],
    linestyle='--',
    color='black',
)
plt.xlabel('Actual transport descriptor')
plt.ylabel('Predicted transport descriptor')
plt.title(f'Actual vs predicted - {best_experiment}, {best_model_name}')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook4_actual_vs_predicted.png', dpi=200)
plt.show()

best_row

### Explanation of the code cell above

This cell selects the best model based on the highest R² score and plots actual versus predicted values.

In an ideal model, all points would lie close to the diagonal dashed line.

- Points near the line indicate accurate predictions.
- Points far from the line indicate larger prediction errors.

This plot is often more intuitive than metrics alone because it shows the quality of predictions visually.


## Feature Importance - Full Model

In [ ]:
rf_full = full_results['Random Forest']['model']

importance_full = pd.DataFrame(
    {
        'feature': full_features,
        'importance': rf_full.feature_importances_,
    }
).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(
    importance_full['feature'][::-1],
    importance_full['importance'][::-1],
)
plt.xlabel('Feature importance')
plt.ylabel('Feature')
plt.title('Random Forest feature importance - full-feature model')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook4_feature_importance_full.png', dpi=200)
plt.show()

importance_full

### Explanation of the code cell above

This cell plots Random Forest feature importance for the full-feature model.

Feature importance estimates how much each feature contributes to the model’s decision-making.

This is a first interpretability step. However, built-in Random Forest importance can be biased toward some feature types, so Notebook 5 later uses permutation importance for a stronger check.


## Feature Importance - Reduced Model

In [ ]:
rf_reduced = reduced_results['Random Forest']['model']

importance_reduced = pd.DataFrame(
    {
        'feature': reduced_features,
        'importance': rf_reduced.feature_importances_,
    }
).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(
    importance_reduced['feature'][::-1],
    importance_reduced['importance'][::-1],
)
plt.xlabel('Feature importance')
plt.ylabel('Feature')
plt.title('Random Forest feature importance - reduced-feature model')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook4_feature_importance_reduced.png', dpi=200)
plt.show()

importance_reduced

### Explanation of the code cell above

This cell plots Random Forest feature importance for the reduced-feature model.

The goal is to see which remaining features are most influential after simplifying the input space.

If the same physical descriptors remain important in both full and reduced models, that strengthens the interpretation.


## Conclusion

In this notebook, we trained machine-learning regression models to predict the transport-related descriptor from physics-informed features extracted from random hydrogen-defect configurations.

Two experimental settings were considered.

First, the full-feature surrogate model used all physically relevant descriptors, including `mean_ipr` and `defect_density`. Because these variables are directly used in the definition of the transport descriptor, this experiment should be interpreted as a surrogate-modeling baseline.

Second, the reduced-feature prediction model removed `mean_ipr` and `defect_density`. This provided a more challenging test of whether the transport proxy can be inferred from indirect defect and spectral descriptors.

The comparison between these two settings is important for scientific interpretation. A high score in the full-feature model confirms that the designed transport proxy can be learned from its defining descriptors. Performance in the reduced-feature model provides a more meaningful estimate of how much indirect physical information is available in defect strength, defect spacing, clustering, spectral bandwidth, and maximum localization.

These results prepare the next stage of the project: feature-importance analysis and optimization of defect configurations.

## Key learning takeaway

After running this notebook, focus on writing one or two sentences in your own words:

- What physical quantity was computed?
- Which feature or parameter changed?
- How did the result affect localization or transport?

This habit will help you turn code execution into scientific understanding.


## Final Formula Reminder

The core logic of this notebook can be summarized as

$$
\text{defect configuration} \rightarrow H \rightarrow \{E_n,\mathbf{c}^{(n)}\} \rightarrow \text{features} \rightarrow \text{prediction or optimization}.
$$

This is the main physics-informed workflow: we start from a material-inspired defect model, convert it into spectral and localization descriptors, and then use those descriptors for machine learning, interpretation, or optimization.
